# Notebook 3 v2 — Train the improved watermark (journal version)

**Changes from v1 (three targeted fixes):**
1. Fake regions in training have many shapes and sizes (ovals, boxes, blobs), not only face ovals,
   so the tamper map outlines the whole edited region (v1 under-marked face swaps).
2. The hidden code is also trained on manipulated images, so it survives a face swap.
3. The checking networks use group normalisation (each image normalised on its own), and every image
   in a batch gets its own random edit, so no per-image normalisation trick is needed at test time.

**What it does:** learns to hide a 32-bit code in a face image so that
- the image still looks the same (PSNR kept at about 36 dB),
- the code survives normal edits (JPEG, resize, blur, noise, colour change),
- a "tamper map" lights up wherever the face was replaced or edited.

**Run it twice, on two accounts at the same time:**
- `MODE = "adaptive"` → texture-adaptive strength (stronger mark in textured areas; ablation)
- `MODE = "fixed"` → same strength everywhere (the final model in the paper uses v2.1 fixed)

**Kaggle settings:** Input: `badasstechie/celebahq-resized-256x256` ·
Accelerator: **GPU P100** · Internet: **On**

**Output (save as a dataset afterwards):** `wm_<MODE>_v2.pt`, `log_<MODE>_v2.csv`, `examples_<MODE>_v2.png`,
`celeba_split.csv` (same split as v1)

In [ ]:
import os, glob, time, random, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips"], check=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm

# ---------------- settings ----------------
MODE = "adaptive"        # "adaptive" (ours) or "fixed" (baseline)
MSG_BITS = 32            # length of the hidden code
BATCH = 16
EPOCHS = 12
LR = 1e-4
TARGET_PSNR = 36.0       # image quality we hold both models at (fair comparison)
PHASE1_ACC = 0.97        # stage 1 (no attacks) lasts until the code is read this well ...
PHASE1_MAX = 6000        # ... or at most this many steps
RAMP_STEPS = 3000        # stage 2: attacks and tamper task are faded in over these steps
BETA = 0.15              # largest allowed change per pixel (images are in [-1, 1])
S_MIN = 0.3              # weakest strength used in smooth areas (adaptive mode)
TIME_LIMIT_H = 11.0      # stop safely before Kaggle's 12-hour limit
OUTDIR = "/kaggle/working"
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| Mode:", MODE)

## Step 1 — Load CelebA-HQ and make a fixed train / val / test split
The same split is reused by later notebooks (`celeba_split.csv`).

In [ ]:
files = sorted(glob.glob("/kaggle/input/**/*.jpg", recursive=True) +
               glob.glob("/kaggle/input/**/*.png", recursive=True))
print("Images found:", len(files))
assert len(files) > 20000, "CelebA-HQ not found - add it as Input"

random.Random(0).shuffle(files)          # fixed shuffle -> same split every time
test_files, val_files, train_files = files[:2000], files[2000:2500], files[2500:]
pd.DataFrame({"path": test_files + val_files + train_files,
              "split": ["test"] * len(test_files) + ["val"] * len(val_files) +
                       ["train"] * len(train_files)}).to_csv(f"{OUTDIR}/celeba_split.csv", index=False)
print(f"train {len(train_files)} | val {len(val_files)} | test {len(test_files)}")

class Faces(Dataset):
    def __init__(self, files, train):
        self.files, self.train = files, train
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert("RGB")
        if img.size != (256, 256):
            img = img.resize((256, 256), Image.BICUBIC)
        x = TF.to_tensor(img) * 2 - 1                 # [-1, 1]
        if self.train and random.random() < 0.5:
            x = x.flip(-1)
        return x

train_dl = DataLoader(Faces(train_files, True), batch_size=BATCH, shuffle=True,
                      num_workers=4, drop_last=True, pin_memory=True)
val_dl = DataLoader(Faces(val_files, False), batch_size=BATCH, shuffle=False, num_workers=4)

## Step 2 — The networks
- **Encoder** (U-Net): image + code → a hidden pattern (residual)
- **Strength map**: how strongly to add the pattern at each pixel
  (adaptive = follows local texture; fixed = 1 everywhere)
- **Decoder** (U-Net): image → the 32-bit code + a tamper map

In [ ]:
def norm_layer(c, norm):
    # "gn" = group normalisation: each image is normalised on its own, so checking one photo
    # behaves exactly like training (no dependence on the rest of the batch)
    return nn.GroupNorm(min(8, c), c) if norm == "gn" else nn.BatchNorm2d(c)

def block(cin, cout, norm="bn"):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), norm_layer(cout, norm), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1), norm_layer(cout, norm), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, cin, cout, ch=(32, 64, 128, 256), norm="bn"):
        super().__init__()
        self.downs = nn.ModuleList()
        c = cin
        for co in ch:
            self.downs.append(block(c, co, norm)); c = co
        self.mid = block(c, c * 2, norm); c = c * 2
        self.bottleneck_ch = c
        self.ups, self.decs = nn.ModuleList(), nn.ModuleList()
        for co in reversed(ch):
            self.ups.append(nn.ConvTranspose2d(c, co, 2, stride=2))
            self.decs.append(block(co * 2, co, norm)); c = co
        self.out = nn.Conv2d(c, cout, 1)

    def forward(self, x):
        skips = []
        for d in self.downs:
            x = d(x); skips.append(x); x = F.max_pool2d(x, 2)
        x = self.mid(x); bott = x
        for up, dec, s in zip(self.ups, self.decs, reversed(skips)):
            x = dec(torch.cat([up(x), s], 1))
        return self.out(x), bott

class Encoder(nn.Module):
    def __init__(self, bits):
        super().__init__()
        self.msg_fc = nn.Linear(bits, 4 * 32 * 32)
        self.unet = UNet(3 + 4, 3)
    def forward(self, x, m):
        mm = self.msg_fc(m * 2 - 1).view(-1, 4, 32, 32)
        mm = F.interpolate(mm, size=x.shape[-2:], mode="nearest")
        r, _ = self.unet(torch.cat([x, mm], 1))
        return torch.tanh(r)

class BitReader(nn.Module):
    """Separate small network that reads the hidden code (does not compete with the tamper map)."""
    def __init__(self, bits, norm="bn"):
        super().__init__()
        layers, c = [], 3
        for co in (32, 32, 64, 64, 128):                 # 256 -> 8 (stride-2 convs)
            layers += [nn.Conv2d(c, co, 3, stride=2, padding=1), norm_layer(co, norm), nn.ReLU(inplace=True)]
            c = co
        self.convs = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(8)
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(c * 64, 512), nn.ReLU(inplace=True), nn.Linear(512, bits))
    def forward(self, x):
        return self.fc(self.pool(self.convs(x)))

class Decoder(nn.Module):
    def __init__(self, bits, norm="gn"):
        super().__init__()
        self.unet = UNet(3, 1, norm=norm)                # tamper map
        self.reader = BitReader(bits, norm=norm)         # hidden code
    def forward(self, x):
        mask, _ = self.unet(x)
        return self.reader(x), mask                      # code logits, tamper-map logits

def texture_map(x):
    """Local contrast of the image, 0 (flat skin/background) .. 1 (hair, eyes, edges)."""
    g = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]).unsqueeze(1)
    mu = F.avg_pool2d(g, 7, 1, 3, count_include_pad=False)
    var = (F.avg_pool2d(g * g, 7, 1, 3, count_include_pad=False) - mu * mu).clamp_min(0)
    t = F.avg_pool2d(var.sqrt(), 9, 1, 4, count_include_pad=False)
    q = torch.quantile(t.flatten(1), 0.95, dim=1).view(-1, 1, 1, 1).clamp_min(1e-4)
    return (t / q).clamp(0, 1)

def strength_map(x, mode=MODE):
    if mode == "fixed":
        return torch.ones_like(x[:, :1])
    return S_MIN + (1 - S_MIN) * texture_map(x)

def embed(enc, x, m, scale=1.0, mode=MODE):
    r = enc(x, m)
    s = strength_map(x, mode)
    return (x + scale * BETA * s * r).clamp(-1, 1), s

## Step 3 — Attacks used during training
- **Normal edits** (the code must survive): JPEG, resize, blur, noise, colour.
- **Manipulation** (the tamper map must find it): a face-shaped or random region is
  replaced with un-watermarked content, like a face swap or an AI edit would do.

In [ ]:
def quantize(x):
    y = ((x + 1) * 127.5).round() / 127.5 - 1
    return x + (y - x).detach()

def jpeg(x, q):
    """Real JPEG in the forward pass, gradient passes straight through."""
    with torch.no_grad():
        xs = ((x.detach().float().clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8).cpu()
        ys = [torchvision.io.decode_jpeg(torchvision.io.encode_jpeg(img, quality=int(q))) for img in xs]
        y = torch.stack(ys).to(x.device).float() / 127.5 - 1
    return x + (y - x).detach()

def resize_attack(x, s):
    h, w = x.shape[-2:]
    small = F.interpolate(x, size=(int(h * s), int(w * s)), mode="bilinear", align_corners=False)
    return F.interpolate(small, size=(h, w), mode="bilinear", align_corners=False)

def benign_attack_each(x, p=1.0):
    """A different random normal edit for every image in the batch."""
    return torch.cat([benign_attack(x[i:i + 1], p) for i in range(len(x))])

def benign_attack(x, p=1.0, kind=None):
    """Apply a random normal edit with probability p (p grows during stage 2)."""
    if kind is None and random.random() >= p:
        return quantize(x)
    k = kind or random.choice(["none", "jpeg", "jpeg", "resize", "blur", "noise", "color"])
    if k == "jpeg":
        return jpeg(x, random.randint(40, 90))
    if k == "resize":
        y = resize_attack(x, random.uniform(0.5, 0.9))
    elif k == "blur":
        sg = random.uniform(0.4, 1.5); y = TF.gaussian_blur(x, [5, 5], [sg, sg])
    elif k == "noise":
        y = x + torch.randn_like(x) * random.uniform(0.005, 0.04)
    elif k == "color":
        y = x * random.uniform(0.85, 1.15) + random.uniform(-0.08, 0.08)
    else:
        y = x
    return quantize(y.clamp(-1, 1))

def random_region_mask(B, H, W, device):
    """Fake regions of many shapes and sizes. Returns soft blend mask and ground-truth mask.
    Each shape is written as a 'distance' d: inside where d < 1."""
    yy, xx = torch.meshgrid(torch.arange(H, device=device, dtype=torch.float32),
                            torch.arange(W, device=device, dtype=torch.float32), indexing="ij")
    ds = []
    for _ in range(B):
        r = random.random()
        if r < 0.30:                  # face-shaped oval (CelebA-HQ faces are centred)
            cx, cy = W / 2 + random.uniform(-15, 15), H * 0.53 + random.uniform(-15, 15)
            rx = random.uniform(45, 90); ry = rx * random.uniform(1.0, 1.4)
            d = ((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2
        elif r < 0.60:                # rotated box (like a face-swap paste region)
            cx, cy = W / 2 + random.uniform(-25, 25), H / 2 + random.uniform(-25, 25)
            a = random.uniform(35, 100); b = a * random.uniform(0.8, 1.25); t = np.deg2rad(random.uniform(-20, 20))
            u = (xx - cx) * np.cos(t) + (yy - cy) * np.sin(t); v = -(xx - cx) * np.sin(t) + (yy - cy) * np.cos(t)
            d = torch.maximum((u / a).abs(), (v / b).abs()) ** 2
        elif r < 0.80:                # irregular blob anywhere (10-45% of the image)
            n = F.interpolate(torch.randn(1, 1, 6, 6, device=device), size=(H, W), mode="bicubic", align_corners=False)[0, 0]
            q = torch.quantile(n.flatten(), random.uniform(0.55, 0.9))
            d = torch.exp(-(n - q) / n.std().clamp_min(1e-6) * 3)        # < 1 where n > q
        else:                         # small local edit anywhere (mouth / eye sized)
            rx = random.uniform(15, 50); ry = rx * random.uniform(0.5, 1.5)
            cx, cy = random.uniform(rx, W - rx), random.uniform(ry, H - ry)
            d = ((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2
        ds.append(d)
    d = torch.stack(ds).unsqueeze(1)
    return torch.sigmoid((1 - d) * 8), (d < 1).float()      # soft blend mask, ground-truth mask

def manipulate(xw, x):
    """Replace a region with un-watermarked content (same face or another face)."""
    B, _, H, W = xw.shape
    soft, hard = random_region_mask(B, H, W, xw.device)
    use_other = (torch.rand(B, 1, 1, 1, device=xw.device) < 0.5).float()
    src = use_other * torch.roll(x, 1, 0) + (1 - use_other) * x
    sg = random.uniform(0.1, 1.0)
    src = TF.gaussian_blur(src, [3, 3], [sg, sg]) * random.uniform(0.95, 1.05)  # generator-like smoothing
    return (xw * (1 - soft) + src * soft).clamp(-1, 1), hard

## Step 4 — Training
Image quality is held at `TARGET_PSNR` automatically: if the image gets too noisy,
the weight on image quality goes up; if it is cleaner than needed, it goes down.

In [ ]:
import lpips
from torchmetrics.functional import structural_similarity_index_measure as ssim_fn

enc, dec = Encoder(MSG_BITS).to(device), Decoder(MSG_BITS).to(device)
opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=LR)
lp = lpips.LPIPS(net="alex", verbose=False).to(device).eval()
for p in lp.parameters():
    p.requires_grad = False
bce = nn.BCEWithLogitsLoss()

def dice_loss(logits, target):
    p = torch.sigmoid(logits)
    inter = (p * target).sum((1, 2, 3))
    return (1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + 1)).mean()

def psnr_per_image(a, b):
    mse = ((a - b) ** 2).flatten(1).mean(1).clamp_min(1e-10)
    return 10 * torch.log10(4 / mse)              # images in [-1, 1] -> range 2

@torch.no_grad()
def validate():
    states = (random.getstate(), torch.random.get_rng_state(),
              torch.cuda.get_rng_state() if device == "cuda" else None)
    random.seed(123); torch.manual_seed(123)
    enc.eval(); dec.eval()
    r = {k: [] for k in ["psnr", "ssim", "lpips", "acc_clean", "acc_jpeg50", "iou", "false_alarm", "false_alarm_jpeg70", "acc_manip"]}
    for x in val_dl:
        x = x.to(device)
        m = torch.randint(0, 2, (x.size(0), MSG_BITS), device=device).float()
        xw = quantize(embed(enc, x, m)[0])
        r["psnr"] += psnr_per_image(xw, x).tolist()
        r["ssim"] += ssim_fn((xw + 1) / 2, (x + 1) / 2, data_range=1.0, reduction="none").tolist()
        r["lpips"] += lp(xw, x).flatten().tolist()
        bits, mb = dec(xw)
        r["acc_clean"] += ((bits > 0).float() == m).float().mean(1).tolist()
        r["false_alarm"] += (torch.sigmoid(mb) > 0.5).float().flatten(1).mean(1).tolist()
        bits_j, _ = dec(jpeg(xw, 50))
        r["acc_jpeg50"] += ((bits_j > 0).float() == m).float().mean(1).tolist()
        _, mj = dec(jpeg(xw, 70))                       # normal checking mode, compressed real image
        r["false_alarm_jpeg70"] += (torch.sigmoid(mj) > 0.5).float().flatten(1).mean(1).tolist()
        xm, gt = manipulate(xw, x)
        bits_mm, mm = dec(quantize(xm))
        r["acc_manip"] += ((bits_mm > 0).float() == m).float().mean(1).tolist()
        pred = (torch.sigmoid(mm) > 0.5).float()
        inter = (pred * gt).sum((1, 2, 3)); union = ((pred + gt) > 0).float().sum((1, 2, 3))
        r["iou"] += (inter / union.clamp_min(1)).tolist()
    random.setstate(states[0]); torch.random.set_rng_state(states[1])
    if states[2] is not None:
        torch.cuda.set_rng_state(states[2])
    enc.train(); dec.train()
    return {k: float(np.mean(v)) for k, v in r.items()}

lam, step, best, log = 1.0, 0, -1e9, []
stage2_start, acc_ema = None, 0.5      # stage 1: learn to hide + read the code; stage 2: add attacks + tamper task
t0 = time.time()
for epoch in range(EPOCHS):
    enc.train(); dec.train()
    pbar = tqdm(train_dl, desc=f"epoch {epoch + 1}/{EPOCHS}")
    for x in pbar:
        x = x.to(device, non_blocking=True)
        m = torch.randint(0, 2, (x.size(0), MSG_BITS), device=device).float()
        xw, _ = embed(enc, x, m)
        mse = F.mse_loss(xw, x)

        if stage2_start is None:                        # stage 1: clean images, code only,
            l_img = torch.zeros((), device=device)      # no quality penalty yet (pattern must be visible to learn)
            bits_b, _ = dec(quantize(xw))
            l_bits = bce(bits_b, m)
            l_mask = torch.zeros((), device=device)
            ramp, target = 0.0, 0.0
        else:                                           # stage 2: fade in quality, attacks, tamper task
            ramp = min(1.0, (step - stage2_start) / RAMP_STEPS)
            target = TARGET_PSNR - 8 * (1 - ramp)       # quality target rises to TARGET_PSNR
            l_img = lam * (100 * mse + lp(xw, x).mean())
            xb = benign_attack_each(xw, p=ramp)         # normal edit (different for each image)
            xm, gt = manipulate(xw, x)                  # malicious edit ...
            xm = benign_attack_each(xm, p=ramp)         # ... then uploaded / compressed
            bits_all, mask_all = dec(torch.cat([xb, xm]))   # real and fake checked in ONE batch
            bits_b, bits_m = bits_all.chunk(2); mask_b, mask_m = mask_all.chunk(2)
            l_bits = bce(bits_b, m) + 0.5 * ramp * bce(bits_m, m)   # code must also survive manipulation
            l_mask = ramp * (bce(mask_b, torch.zeros_like(mask_b)) + bce(mask_m, gt) + dice_loss(mask_m, gt))
        loss = l_img + 2 * l_bits + l_mask

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        with torch.no_grad():
            psnr = 10 * torch.log10(4 / mse.clamp_min(1e-10)).item()
            if stage2_start is not None:
                lam = float(np.clip(lam * (1.01 if psnr < target else 0.99), 0.01, 100))
            acc = ((bits_b > 0).float() == m).float().mean().item()
            acc_ema = 0.98 * acc_ema + 0.02 * acc
        step += 1
        if stage2_start is None and (acc_ema >= PHASE1_ACC or step >= PHASE1_MAX):
            stage2_start = step
            print(f"\n>>> Stage 2 starts at step {step} (code accuracy {acc_ema:.3f})")
        if step % 50 == 0:
            pbar.set_postfix(stage=1 if stage2_start is None else 2, psnr=f"{psnr:.1f}",
                             bit_acc=f"{acc_ema:.3f}", mask=f"{l_mask.item():.3f}", lam=f"{lam:.2f}")
        if time.time() - t0 > TIME_LIMIT_H * 3600:
            break

    v = validate()
    v.update(epoch=epoch + 1, step=step, lam=lam, minutes=(time.time() - t0) / 60)
    log.append(v)
    pd.DataFrame(log).to_csv(f"{OUTDIR}/log_{MODE}_v2.csv", index=False)
    print({k: round(val, 4) for k, val in v.items()})

    ckpt = dict(enc=enc.state_dict(), dec=dec.state_dict(), mode=MODE, bits=MSG_BITS, dec_norm="gn", version=2,
                beta=BETA, s_min=S_MIN, epoch=epoch + 1, val=v)
    torch.save(ckpt, f"{OUTDIR}/wm_{MODE}_v2_last.pt")
    score = v["acc_jpeg50"] + v["iou"] + 0.5 * v["acc_manip"] - v["false_alarm"] - v["false_alarm_jpeg70"]
    fully_trained = stage2_start is not None and step - stage2_start >= RAMP_STEPS
    if fully_trained and score > best and v["psnr"] > TARGET_PSNR - 1.0:
        best = score
        torch.save(ckpt, f"{OUTDIR}/wm_{MODE}_v2.pt")
        print(f"  saved best model (score {score:.4f})")
    if time.time() - t0 > TIME_LIMIT_H * 3600:
        print("Time limit reached - stopping")
        break

if not os.path.exists(f"{OUTDIR}/wm_{MODE}_v2.pt"):
    torch.save(ckpt, f"{OUTDIR}/wm_{MODE}_v2.pt")

## Step 5 — Look at examples (draft paper figure)

In [ ]:
import matplotlib.pyplot as plt

ck = torch.load(f"{OUTDIR}/wm_{MODE}_v2.pt", map_location=device)
enc.load_state_dict(ck["enc"]); dec.load_state_dict(ck["dec"]); enc.eval(); dec.eval()
print("Best epoch:", ck["epoch"], ck["val"])

with torch.no_grad():
    random.seed(7); torch.manual_seed(7)
    x = next(iter(val_dl))[:4].to(device)
    m = torch.randint(0, 2, (4, MSG_BITS), device=device).float()
    xw, s = embed(enc, x, m)
    xw = quantize(xw)
    xm, gt = manipulate(xw, x)
    _, mm = dec(quantize(xm))

show = lambda t: ((t.clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()
cols = ["Original", "Watermarked", "Difference x10", "Strength map", "Manipulated", "True region", "Predicted tamper map"]
fig, ax = plt.subplots(4, len(cols), figsize=(2.2 * len(cols), 9))
for i in range(4):
    imgs = [show(x[i]), show(xw[i]), ((xw[i] - x[i]).abs() * 10).clamp(0, 1).mean(0).cpu().numpy(),
            s[i, 0].cpu().numpy(), show(xm[i]), gt[i, 0].cpu().numpy(), torch.sigmoid(mm[i, 0]).cpu().numpy()]
    for j, im in enumerate(imgs):
        ax[i, j].imshow(im, cmap=None if im.ndim == 3 else "gray", vmin=0, vmax=1)
        ax[i, j].axis("off")
        if i == 0:
            ax[i, j].set_title(cols[j], fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTDIR}/examples_{MODE}_v2.png", dpi=150)
plt.show()